# scDRS-FM toy example — Rheumatoid arthritis on Tabula Muris Senis (FACS)

This notebook is a small, self-contained walk-through of **scDRS-FM**: it scores every
cell for its relevance to a disease (here **rheumatoid arthritis**, an autoimmune trait),
then uses the **conditional (metacell) analysis** to pull out *independent cell populations*
that carry the disease signal.

The plotting here follows the same conventions we use in the full project analysis
(`plotting/`): a **two-panel UMAP** where the left panel shows the per-cell conditional
disease score and the right panel shows the **independent populations**, defined as the
cells that are significant in **both** the marginal (per-cell) and conditional
(per-metacell) analyses — i.e. the *marginal × conditional* cells.

**What you need to reproduce this** (all shipped under `examples/data/`):

| File | What it is |
|---|---|
| `data/toy_tms_facs.h5ad` | ~2,000 immune cells from Tabula Muris Senis FACS, raw counts (gzip-compressed) |
| `data/toy_tms_facs.cov` | matching covariates (`const, n_genes, sex_male, age`) |
| `data/gs/PASS_Rheumatoid_Arthritis` | MAGMA z-score weighted gene set for RA (human symbols) |
| `data/out/*.gz` | precomputed scDRS-FM output (so the notebook runs instantly) |

The toy is deterministic; `make_toy_data.py` regenerates the exact same subset.

## 0. Imports and configuration

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Optional, Sequence

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from statsmodels.stats.multitest import multipletests

sns.set_context("notebook")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.family"] = ["Liberation Sans", "Arimo", "DejaVu Sans"]
plt.rcParams["svg.fonttype"] = "none"  # keep SVG text editable

DATA = Path("data")
OUT = DATA / "out"          # precomputed scDRS-FM output
GS = DATA / "gs"            # gene set(s)
TRAIT = "PASS_Rheumatoid_Arthritis"
TRAIT_LABEL = "Rheumatoid arthritis"
CELLPOP_KEY = "cell_ontology_class"   # TMS_FACS cell-type annotation (the "cell population")
FDR_ALPHA = 0.1                        # BH-FDR threshold used throughout
print("scanpy", sc.__version__)

scanpy 1.10.4


## 1. Load the toy data

The `.h5ad` holds **raw counts** for ~2,000 cells drawn from the immune-rich tissues of
Tabula Muris Senis FACS (Marrow, Spleen, Thymus, Lung, Limb_Muscle). scDRS-FM does its own
normalisation internally, so raw counts are exactly what it expects.

In [2]:
adata = sc.read_h5ad(DATA / "toy_tms_facs.h5ad")
cov = pd.read_csv(DATA / "toy_tms_facs.cov", sep="\t", index_col=0)
assert list(adata.obs_names) == list(cov.index), "cov must be aligned to the h5ad"

print(f"toy AnnData: {adata.n_obs} cells x {adata.n_vars} genes")
print(f"file is gzip-compressed and stores raw integer counts as float32\n")
print("cells per tissue:")
print(adata.obs["tissue"].value_counts())
print("\ntop cell types (cell_ontology_class):")
print(adata.obs[CELLPOP_KEY].value_counts().head(8))

toy AnnData: 1999 cells x 20326 genes
file is gzip-compressed and stores raw integer counts as float32

cells per tissue:
tissue
Marrow         916
Lung           330
Spleen         267
Thymus         260
Limb_Muscle    226
Name: count, dtype: int64

top cell types (cell_ontology_class):
cell_ontology_class
B cell                            242
hematopoietic stem cell           193
granulocyte                       192
naive B cell                      177
bronchial smooth muscle cell      144
DN4 thymocyte                     119
skeletal muscle satellite cell    103
thymocyte                         101
Name: count, dtype: int64


## 2. How the scDRS-FM output was produced

The `data/out/*.gz` files are the output of running scDRS-FM once, from the repository root:

```bash
python run_scdrs_fm.py \
    examples/data/toy_tms_facs.h5ad \
    examples/data/toy_tms_facs.cov \
    examples/data/out \
    examples/data/gs \
    PASS_Rheumatoid_Arthritis \
    --h5ad_species mouse \
    --imputation magic \
    --flag_raw_count \
    --flag_filter
```

On this toy that finishes in **under 30 seconds** on a laptop. It writes two files we use below:

* `PASS_Rheumatoid_Arthritis.marginal_score.gz` — **one row per cell** (`norm_score`, `pval`, `metacell`, …)
* `PASS_Rheumatoid_Arthritis.conditional.tagging_score.gz` — **one row per metacell**
  (`cell_ids`, `independent_signal`, `norm_score`, `pval`, …)

> The compressed toy produces **bit-for-bit identical** scores to the original uncompressed
> object — dropping all-zero genes and gzip-compressing the counts changes nothing scDRS-FM sees.

In [3]:
df_marg = pd.read_csv(OUT / f"{TRAIT}.marginal_score.gz", sep="\t", index_col=0)
df_cond = pd.read_csv(OUT / f"{TRAIT}.conditional.tagging_score.gz", sep="\t", index_col=0)

print("marginal (per-cell):", df_marg.shape, "->", list(df_marg.columns))
display(df_marg.head(3))
print("\nconditional (per-metacell):", df_cond.shape, "->", list(df_cond.columns))
display(df_cond.head(3))

marginal (per-cell): (1999, 7) -> ['raw_score', 'norm_score', 'mc_pval', 'pval', 'nlog10_pval', 'zscore', 'metacell']


,raw_score,norm_score,mc_pval,pval,nlog10_pval,zscore,metacell
index,,,,,,,
A12_B002452_B009020_S12.mm10-plus-0-0,0.557584,2.760219,0.001998,0.004496,2.347150,2.612340,119
A13_B002452_B009020_S13.mm10-plus-0-0,0.611480,4.518847,0.000999,0.000049,4.309587,3.895370,121
A17_B002452_B009020_S17.mm10-plus-0-0,0.551035,2.183523,0.007992,0.017110,1.766762,2.117481,114



conditional (per-metacell): (158, 9) -> ['cell_ids', 'metacell_size', 'independent_signal', 'raw_score', 'norm_score', 'mc_pval', 'pval', 'nlog10_pval', 'zscore']


,cell_ids,metacell_size,independent_signal,raw_score,norm_score,mc_pval,pval,nlog10_pval,zscore
0,"A13_B000268_B009896_S13.mm10-plus-4-0,C13_B002...",31,105,0.012412,1.421145,0.080919,0.081854,1.086961,1.392709
1,"A1_B002819_B009892_S1.mm10-plus-4-0,A21_B00281...",30,131,-0.013835,-1.314763,0.923077,0.911323,0.040328,-1.348949
2,"E20_D045342_B009019_S116.mm10-plus-0-0,O14_D04...",30,157,0.034482,2.520782,0.011988,0.008342,2.078744,2.393611


## 3. Analysis + plotting helpers

These are the exact utilities used in the full project analysis notebook
(`plotting/`), copied here verbatim so the toy is self-contained. The important ones are:

* **`assign_conditional_scores_to_cells_filtered_signals`** — reads the conditional
  (per-metacell) file, spreads each metacell's conditional score back onto its member cells,
  and defines the **independent populations** as the *intersection* of marginal-significant
  cells and conditional-significant cells (the **marginal × conditional** cells). Populations
  are then filtered to those with enough causal cells that also make up a meaningful fraction
  of at least one cell type.
* **`plot_ibd_scores_and_signals`** — the two-panel disease-score / independent-population UMAP.

In [4]:
def pick_first_existing_col(df, candidates, *, what):
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"{what}: none of these columns exist: {tuple(candidates)}")


def bh_fdr_mask(pvals, alpha=0.05):
    # Benjamini-Hochberg FDR reject mask, ignoring NaN/inf values.
    p = np.asarray(pvals, dtype=float)
    ok = np.isfinite(p)
    out = np.zeros(len(p), dtype=bool)
    if ok.sum() == 0:
        return out
    reject, _, _, _ = multipletests(p[ok], alpha=alpha, method="fdr_bh")
    out[ok] = reject
    return out


def descending_score_order(values, mask=None):
    # Point indices ordered by score from highest to lowest (NaNs last).
    values_arr = np.asarray(values, dtype=float)
    indices = np.arange(values_arr.size)
    if mask is not None:
        mask_arr = np.asarray(mask, dtype=bool)
        indices = indices[mask_arr]
    if indices.size == 0:
        return indices
    sortable = np.where(np.isnan(values_arr[indices]), -np.inf, values_arr[indices])
    return indices[np.argsort(-sortable, kind="mergesort")]


def clean_umap_axis(ax):
    # Remove axes, ticks, and spines from a UMAP axis.
    ax.set_xlabel(""); ax.set_ylabel("")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_frame_on(False)
    for spine in ax.spines.values():
        spine.set_visible(False)


def explode_cell_ids(df, *, cell_ids_col):
    # Explode comma-separated cell_ids from scDRS-FM conditional rows.
    long = df.copy()
    long["cell_id"] = long[cell_ids_col].astype(str).str.split(",")
    long = long.explode("cell_id", ignore_index=True)
    long["cell_id"] = long["cell_id"].astype(str).str.strip()
    long = long[(long["cell_id"] != "") & (long["cell_id"] != "nan")]
    return long

In [5]:
def prepare_umap(adata, *, n_neighbors=15, n_pcs=40, max_pcs=50, force=False):
    # Return a copy with X_umap present (standard scanpy HVG -> scale -> PCA -> UMAP).
    adata_umap = adata.copy()
    if ("X_umap" in adata_umap.obsm) and not force:
        return adata_umap
    sc.pp.highly_variable_genes(
        adata_umap, subset=False, min_disp=0.5, min_mean=0.0125,
        max_mean=10, n_bins=20, n_top_genes=None,
    )
    sc.pp.scale(adata_umap, max_value=10, zero_center=False)
    sc.pp.pca(
        adata_umap, n_comps=min(adata_umap.n_obs, max_pcs),
        use_highly_variable=True, svd_solver="arpack",
    )
    sc.pp.neighbors(
        adata_umap, n_neighbors=min(adata_umap.n_obs, n_neighbors),
        n_pcs=min(adata_umap.n_obs, n_pcs),
    )
    sc.tl.umap(adata_umap)
    return adata_umap

In [6]:
def assign_conditional_scores_to_cells_filtered_signals(
    *,
    adata_umap: sc.AnnData,
    out_folder: Path,
    trait: str,
    indep_sig_col: str = "independent_signal",
    cell_ids_col: str = "cell_ids",
    score_col_candidates: Sequence[str] = ("tagging_score", "conditional_score", "score", "z", "zscore", "stat"),
    pval_col_candidates_cond: Sequence[str] = ("pval", "mc_pval"),
    pval_col_candidates_marg: Sequence[str] = ("pval", "mc_pval"),
    cond_file_suffix: str = ".conditional.tagging_score.gz",
    marg_file_suffix: str = ".marginal_score.gz",
    marginal_metacell_col: str = "metacell",
    fdr_alpha: float = 0.1,
    min_causal_cells_per_signal: int = 100,
    cellpop_key: str = "Cell_population",
    min_fraction_within_any_cellpop: float = 0.05,
    obs_score_key: Optional[str] = None,
    obs_sig_key: Optional[str] = None,
    overwrite: bool = True,
):
    """
    Assign one conditional disease score per cell and filtered independent-population IDs.

    A retained independent population must have enough marginal INTERSECT conditional cells
    and account for at least `min_fraction_within_any_cellpop` of one cell population.
    """
    prefix = Path(trait).name
    out_folder = Path(out_folder)
    cond_file = out_folder / f"{prefix}{cond_file_suffix}"
    marg_file = out_folder / f"{prefix}{marg_file_suffix}"

    df_cond = pd.read_csv(cond_file, sep="\t", compression="infer", index_col=0)
    df_marg = pd.read_csv(marg_file, sep="\t", compression="infer", index_col=0)

    if cellpop_key not in adata_umap.obs.columns:
        raise ValueError(f"adata_umap.obs missing {cellpop_key!r}")
    if cell_ids_col not in df_cond.columns or indep_sig_col not in df_cond.columns:
        raise ValueError(f"{cond_file} must contain {cell_ids_col!r} and {indep_sig_col!r}")
    if marginal_metacell_col not in df_marg.columns:
        raise ValueError(f"{marg_file} missing {marginal_metacell_col!r}")

    score_col = pick_first_existing_col(df_cond, score_col_candidates, what="conditional score")
    pcol_cond = pick_first_existing_col(df_cond, pval_col_candidates_cond, what="conditional p-value")
    pcol_marg = pick_first_existing_col(df_marg, pval_col_candidates_marg, what="marginal p-value")

    obs_names = adata_umap.obs_names.astype(str)

    marg_sig_mask = bh_fdr_mask(df_marg[pcol_marg].to_numpy(), alpha=fdr_alpha)
    marg_sig_cells = obs_names.intersection(df_marg.index[marg_sig_mask].astype(str))

    cond_sig_mask = bh_fdr_mask(df_cond[pcol_cond].to_numpy(), alpha=fdr_alpha)
    df_cond_sig = df_cond.loc[cond_sig_mask] if cond_sig_mask.any() else df_cond.iloc[0:0]

    long_all = explode_cell_ids(df_cond[[score_col, indep_sig_col, cell_ids_col]], cell_ids_col=cell_ids_col)
    long_all = long_all[long_all["cell_id"].isin(obs_names)]

    long_sig = (
        explode_cell_ids(df_cond_sig[[score_col, indep_sig_col, cell_ids_col]], cell_ids_col=cell_ids_col)
        if len(df_cond_sig) else long_all.iloc[0:0].copy()
    )
    long_sig = long_sig[long_sig["cell_id"].isin(obs_names)]

    duplicated = long_all["cell_id"].duplicated(keep=False)
    if duplicated.any():
        examples = long_all.loc[duplicated, ["cell_id", indep_sig_col]].head(10)
        raise ValueError(f"A cell appears in multiple conditional metacells. Examples:\n{examples}")

    long_sig[indep_sig_col] = pd.to_numeric(long_sig[indep_sig_col], errors="coerce")
    long_sig = long_sig[long_sig[indep_sig_col].notna()]
    long_sig[indep_sig_col] = long_sig[indep_sig_col].astype(int)
    long_sig = long_sig[long_sig[indep_sig_col] >= 0]

    causal_cells_by_sig: dict[int, pd.Index] = {}
    causal_counts: dict[int, int] = {}
    for sig, sub in long_sig.groupby(indep_sig_col, sort=True):
        cond_cells = pd.Index(sub["cell_id"].astype(str)).unique()
        # Independent population = cells significant in BOTH marginal and conditional analyses.
        causal_cells = marg_sig_cells.intersection(cond_cells)
        sig = int(sig)
        causal_cells_by_sig[sig] = causal_cells
        causal_counts[sig] = int(len(causal_cells))

    cellpop = adata_umap.obs[cellpop_key].astype(str).copy()
    cellpop.index = obs_names
    cellpop_totals = cellpop.value_counts()

    max_frac_by_sig: dict[int, float] = {}
    for sig, causal_cells in causal_cells_by_sig.items():
        if len(causal_cells) == 0:
            max_frac_by_sig[sig] = 0.0
            continue
        counts_in_sig = cellpop.loc[cellpop.index.intersection(causal_cells)].value_counts()
        if counts_in_sig.empty:
            max_frac_by_sig[sig] = 0.0
            continue
        max_frac_by_sig[sig] = float((counts_in_sig / cellpop_totals.loc[counts_in_sig.index]).max())

    kept_old_sigs = sorted(
        s for s, n in causal_counts.items()
        if n >= min_causal_cells_per_signal and max_frac_by_sig.get(s, 0.0) >= min_fraction_within_any_cellpop
    )
    remap = {old: idx + 1 for idx, old in enumerate(kept_old_sigs)}

    score_key = obs_score_key or f"{prefix}_conditional_score"
    sig_key = obs_sig_key or f"{prefix}_{indep_sig_col}_filtered"

    score_s = pd.Series(np.nan, index=obs_names, dtype=float)
    score_s.loc[long_all["cell_id"].values] = pd.to_numeric(long_all[score_col].values, errors="coerce")

    sig_s = pd.Series(-1, index=obs_names, dtype=int)
    for old_sig in kept_old_sigs:
        causal_cells = causal_cells_by_sig.get(old_sig, pd.Index([], dtype=str))
        if len(causal_cells):
            sig_s.loc[obs_names.intersection(causal_cells)] = remap[old_sig]

    adata_umap.obs[score_key] = score_s.reindex(adata_umap.obs_names).values
    adata_umap.obs[sig_key] = pd.Categorical(
        sig_s.reindex(adata_umap.obs_names).astype(int),
        categories=[-1] + list(range(1, len(kept_old_sigs) + 1)),
        ordered=True,
    )
    return df_cond, df_marg, remap

In [7]:
def plot_ibd_scores_and_signals(
    adata_umap: sc.AnnData,
    score_key: str,
    sig_key: str,
    *,
    trait_label: str = "IBD",
    cellpop_key: str = "Cell_population",
    notsig_label: str = "Not sig.",
    base_size: float = 6,
    causal_size: float = 20,
    signal_size: float = 16,
    other_alpha: float = 0.3,
    cmap: str = "RdBu_r",
    center_at: float = 0.0,
    composition_min_ratio: float = 0.01,
    composition_min_cells: int = 50,
    sort_descending: bool = False,
    savepath=None,
    dpi: int = 300,
    show: bool = True,
):
    """Two-panel trait UMAP: scDRS-FM conditional disease scores and independent populations."""
    if "X_umap" not in adata_umap.obsm:
        raise ValueError("adata_umap.obsm['X_umap'] not found. Run prepare_umap first.")
    for required in [score_key, sig_key, cellpop_key]:
        if required not in adata_umap.obs:
            raise KeyError(f"{required!r} not found in adata_umap.obs")

    U = adata_umap.obsm["X_umap"]
    x, y = U[:, 0], U[:, 1]
    scores = pd.to_numeric(adata_umap.obs[score_key], errors="coerce").to_numpy()
    sig_raw = (
        pd.to_numeric(adata_umap.obs[sig_key].astype(str), errors="coerce")
        .fillna(-1).astype(int).to_numpy()
    )
    causal = sig_raw != -1
    cell_pops = adata_umap.obs[cellpop_key].astype(str).fillna("NA")
    total_by_celltype = cell_pops.value_counts()

    kept = sorted(np.unique(sig_raw[causal]).tolist())

    # Wider figure + explicit spacing so the two titles clear each other and the
    # right-hand external legend has room (long trait labels need more width than "IBD").
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.4))
    fig.subplots_adjust(left=0.02, right=0.72, wspace=0.12, top=0.90, bottom=0.03)

    finite = np.isfinite(scores)
    max_abs = float(np.nanmax(np.abs(scores[finite] - center_at))) if finite.any() else 1.0
    if max_abs == 0:
        max_abs = 1.0
    norm = mcolors.TwoSlopeNorm(vcenter=center_at, vmin=center_at - max_abs, vmax=center_at + max_abs)

    background_order = descending_score_order(scores, ~causal) if sort_descending else np.flatnonzero(~causal)
    causal_order = descending_score_order(scores, causal) if sort_descending else np.flatnonzero(causal)

    # LEFT: conditional disease scores, causal cells outlined
    ax = axes[0]
    ax.scatter(x[background_order], y[background_order], c=scores[background_order],
               cmap=cmap, norm=norm, s=base_size, alpha=other_alpha, linewidths=0)
    ax.scatter(x[causal_order], y[causal_order], c=scores[causal_order],
               cmap=cmap, norm=norm, s=causal_size, alpha=0.6, edgecolors="black", linewidths=0.4)
    ax.set_title(f"scDRS-FM disease scores ({trait_label})", fontsize=15)
    sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)

    # RIGHT: independent populations, coloured, with cell-type composition in the legend
    ax2 = axes[1]
    notsig_color = "#D0D0D0"
    ax2.scatter(x[background_order], y[background_order], color=notsig_color,
                s=base_size, alpha=other_alpha, linewidths=0)
    if len(kept) <= 10:
        palette = sns.color_palette("tab10", n_colors=max(1, len(kept))).as_hex()
    elif len(kept) <= 20:
        palette = sns.color_palette("tab20", n_colors=len(kept)).as_hex()
    else:
        palette = sns.color_palette("husl", n_colors=len(kept)).as_hex()
    sig_to_color = {s: palette[i] for i, s in enumerate(kept)}

    handles = [Line2D([0], [0], marker="o", linestyle="None", markerfacecolor=notsig_color,
                      markeredgecolor="black", markeredgewidth=0.8, markersize=7,
                      alpha=other_alpha, label=notsig_label)]

    for s in kept:
        mask = sig_raw == s
        signal_order = descending_score_order(scores, mask) if sort_descending else np.flatnonzero(mask)
        ax2.scatter(x[signal_order], y[signal_order], color=sig_to_color[s],
                    s=signal_size, alpha=0.6, edgecolors="black", linewidths=0.4)

        signal_cell_counts = cell_pops[mask].value_counts()
        comp_rows = []
        for ct, in_signal in signal_cell_counts.items():
            total_count = int(total_by_celltype.get(ct, 0))
            if total_count == 0:
                continue
            ratio = in_signal / total_count
            if ratio > composition_min_ratio and in_signal >= composition_min_cells:
                comp_rows.append((ct, int(in_signal), total_count, ratio))
        comp_rows = sorted(comp_rows, key=lambda z: (-z[3], -z[1], z[0]))
        if comp_rows:
            comp_text = "\n".join(f"   {ct}: {in_signal}/{total_count}"
                                  for ct, in_signal, total_count, _ in comp_rows)
            label = f"Indep. population {s}\n{comp_text}"
        else:
            label = f"Indep. population {s}"
        handles.append(Line2D([0], [0], marker="o", linestyle="None",
                              markerfacecolor=sig_to_color[s], markeredgecolor="black",
                              markersize=8, label=label))

    ax2.set_title(f"scDRS-FM independent populations ({trait_label})", fontsize=15)
    legend = ax2.legend(handles=handles, frameon=False, loc="center left",
                        bbox_to_anchor=(1.02, 0.5), fontsize=12, handletextpad=0.8,
                        labelspacing=1.2, borderaxespad=0.0)
    for text_item in legend.get_texts():
        text_item.set_fontsize(12); text_item.set_multialignment("left"); text_item.set_fontfamily("monospace")

    for axis in axes:
        clean_umap_axis(axis)

    if savepath is not None:
        savepath = Path(savepath)
        savepath.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(savepath, dpi=dpi, bbox_inches="tight")
        # also save an editable SVG alongside the PNG
        if savepath.suffix.lower() == ".png":
            fig.savefig(savepath.with_suffix(".svg"), bbox_inches="tight")
        print(f"Saved {savepath}")
    if show:
        plt.show()
    else:
        plt.close(fig)

## 4. Build the UMAP embedding

The toy ships raw counts and no embedding, so we compute one for visualisation only
(the scores were already computed from raw counts). We normalise and log-transform exactly
as in the project analysis (`normalize_per_cell` to 1e4, then `log1p`) before `prepare_umap`.

In [8]:
adata_umap = adata.copy()
sc.pp.normalize_per_cell(adata_umap, counts_per_cell_after=1e4)
sc.pp.log1p(adata_umap)
adata_umap = prepare_umap(adata_umap, force=True)
print("UMAP ready:", adata_umap.obsm["X_umap"].shape)

/workspace/scdrsfm_env/lib/python3.11/site-packages/scanpy/preprocessing/_pca.py:374: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  warn(msg, FutureWarning)


UMAP ready: (1999, 2)


## 5. Identify the independent populations (marginal × conditional cells)

`assign_conditional_scores_to_cells_filtered_signals` does three things:

1. spreads each metacell's **conditional** disease score back onto its member cells
   (written to `obs["<trait>_conditional_score"]`);
2. flags cells significant in **both** the marginal and conditional analyses — the
   **marginal × conditional** cells — grouped by `independent_signal`;
3. keeps only the populations with enough of those causal cells that also make up a
   meaningful fraction of at least one cell type.

The full project analysis uses `min_causal_cells_per_signal=100` on the ~30k-cell Soskic
data. This toy has only ~2,000 cells, so we scale that threshold down to **30** (and the
cell-population fraction to **0.02**) so the smaller-but-real populations survive.

In [9]:
df_cond, df_marg, remap = assign_conditional_scores_to_cells_filtered_signals(
    adata_umap=adata_umap,
    out_folder=OUT,
    trait=TRAIT,
    fdr_alpha=FDR_ALPHA,
    cellpop_key=CELLPOP_KEY,
    min_causal_cells_per_signal=30,     # scaled down for the ~2k-cell toy
    min_fraction_within_any_cellpop=0.02,
)

score_key = f"{TRAIT}_conditional_score"
sig_key = f"{TRAIT}_independent_signal_filtered"

n_marg_sig = int(bh_fdr_mask(df_marg["pval"].to_numpy(), FDR_ALPHA).sum())
n_cond_sig = int(bh_fdr_mask(df_cond["pval"].to_numpy(), FDR_ALPHA).sum())
n_causal = int((adata_umap.obs[sig_key].astype(int) != -1).sum())
print(f"marginal-significant cells (FDR<{FDR_ALPHA}): {n_marg_sig}")
print(f"conditional-significant metacells (FDR<{FDR_ALPHA}): {n_cond_sig} / {df_cond.shape[0]}")
print(f"retained independent populations: {len(remap)}  (raw id -> plot id: {remap})")
print(f"total marginal x conditional causal cells: {n_causal}")

# cell-type composition of each retained population
for plot_id in sorted(x for x in adata_umap.obs[sig_key].astype(int).unique() if x != -1):
    m = adata_umap.obs[sig_key].astype(int) == plot_id
    top = adata_umap.obs.loc[m, CELLPOP_KEY].value_counts().head(4)
    print(f"\nindependent population {plot_id}: {int(m.sum())} cells")
    print(top.to_string())

marginal-significant cells (FDR<0.1): 619
conditional-significant metacells (FDR<0.1): 18 / 158
retained independent populations: 2  (raw id -> plot id: {7: 1, 157: 2})
total marginal x conditional causal cells: 207

independent population 1: 71 cells
cell_ontology_class
DN4 thymocyte                      67
CD4-positive, alpha-beta T cell     2
epithelial cell of thymus           1
thymocyte                           1

independent population 2: 136 cells
cell_ontology_class
NK cell                            37
CD8-positive, alpha-beta T cell    25
mature alpha-beta T cell           25
DN4 thymocyte                      22


## 6. The two-panel figure

* **Left** — every cell coloured by its scDRS-FM **conditional disease score**
  (`RdBu_r`, centred at 0); the **marginal × conditional causal cells** are drawn larger with a
  black outline.
* **Right** — the same UMAP with each **independent population** in its own colour; the legend
  gives the cell-type composition (cells-in-population / total-of-that-type).

In [10]:
plot_ibd_scores_and_signals(
    adata_umap,
    score_key,
    sig_key,
    trait_label=TRAIT_LABEL,
    cellpop_key=CELLPOP_KEY,
    composition_min_ratio=0.01,
    composition_min_cells=10,   # scaled down for the toy
    other_alpha=0.3,
    signal_size=16,
    sort_descending=True,
    savepath="fig_scores_and_independent_populations.png",
    show=True,
)

Saved fig_scores_and_independent_populations.png


## 7. What the figure shows

On this immune toy, rheumatoid arthritis resolves into a few **independent populations** that
map onto distinct T/NK compartments — mirroring the biology in the full analysis:

* a **peripheral lymphocyte** population (NK cells + mature αβ / CD8⁺ T cells across
  marrow, spleen and lung), and
* a largely **thymic developmental T-cell** population (DN4 thymocytes).

The left panel shows the disease score is broadly elevated across lymphocytes, while the
conditional analysis on the right **separates** that signal into populations that are
independently associated with RA rather than one diffuse blob. That marginal → conditional
step — keeping only cells significant in *both* analyses — is the core idea this notebook
demonstrates, in the same format used throughout the project.